In [3]:
!nvidia-smi

Thu Oct 15 10:49:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P0             29W /   70W |     355MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 1. Verify GPU connection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device:- {device}")
if device.type == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# 2. Define a simple neural network
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

net = Net().to(device)

# 3. Create dummy data to test training loop (Batch of 64, 784 features)
inputs = torch.randn(640, 784)
targets = torch.randint(0, 10, (640,))
dataset = TensorDataset(inputs, targets)
data_loader = DataLoader(dataset, batch_size=64, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.01)

# 4. Run a quick training epoch
net.train()
for epoch in range(2):
    running_loss = 0.0
    for batch_idx, (batch_inputs, batch_targets) in enumerate(data_loader):
        # Move data to the active device (GPU)
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)
        
        optimizer.zero_grad()
        output = net(batch_inputs)
        loss = criterion(output, batch_targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f"Epoch {epoch+1} finished. Loss: {running_loss / len(data_loader):.4f}")

print("Colab GPU execution test passed successfully!")

Using device:- cuda
GPU Name: Tesla T4
Epoch 1 finished. Loss: 2.3264
Epoch 2 finished. Loss: 2.3008
Colab GPU execution test passed successfully!


In [3]:
import time
import torch

# 1. Automatically detect the best available device (CUDA/Colab GPU if connected, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Targeting device automatically: {device}")

if device.type == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

# 2. Define the benchmark workload
def run_benchmark(device, iterations=50):
    size = 4096
    print(f"Allocating matrices of size {size}x{size} on {device}...")
    
    a = torch.randn(size, size, device=device)
    b = torch.randn(size, size, device=device)
    
    # Warm-up pass
    _ = torch.matmul(a, b)
    if device.type == "cuda":
        torch.cuda.synchronize()
        
    start_time = time.time()
    for _ in range(iterations):
        _ = torch.matmul(a, b)
        if device.type == "cuda":
            torch.cuda.synchronize()
            
    elapsed = time.time() - start_time
    print(f"Completed {iterations} matrix multiplications in {elapsed:.4f} seconds.\n")
    return elapsed

# 3. Execute automatically based on whichever kernel/environment is active
elapsed_time = run_benchmark(device)

Targeting device automatically: cuda
GPU Model: Tesla T4
Allocating matrices of size 4096x4096 on cuda...
Completed 50 matrix multiplications in 1.7163 seconds.



In [ ]:
# Model size estimation
#GPU_size = torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else 0 #in Bytes
GPU_size = 24 * 1024 ** 3 #in Bytes
fp16 = 2 #in Bytes
fp32 = 4 #in Bytes

max_fp16_elements = GPU_size // fp16
max_fp32_elements = GPU_size // fp32

print(f"GPU Memory: {GPU_size / (1024 ** 3):.2f} GB = {GPU_size / (1024 ** 2):.2f} MB = {GPU_size / 1024:.2f} Kb = {GPU_size} Bytes")
print(f"Max FP16 elements: {max_fp16_elements * 10 ** -9:.2f} billion")
print(f"Max FP32 elements: {max_fp32_elements * 10 ** -9:.2f} billion")
print("Estimation complete.")


GPU Memory: 24.00 GB = 24576.00 MB = 25165824.00 Kb = 25769803776 Bytes
Max FP16 elements: 12.88 billion
Max FP32 elements: 6.44 billion
Estimation complete.
77.8 ns ± 1.88 ns per loop (mean ± std. dev. of 7 runs, 10000000 loops each)


In [ ]:
%timeit max_fp16_elements = GPU_size // fp16

$E = mc^2$